# RB (Running Back) round regression (Ridge)

Predict draft round 1–8 (8 = undrafted) for running backs using combine + RAS + PFF rushing efficiency metrics.

- **Train**: 2015–2023 from `rb_training.csv` (RB/data_cleaning.py has already merged RAS and PFF rushing).
- **Test**: `rb_testing.csv` filtered to 2024/2025 (drafted only, once `rb_drafted_20xx.csv` exist); 2026 from `rb_drafted_2026.csv`.

In [42]:
import numpy as np
import pandas as pd
import os
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# RB feature set: combine + RAS + PFF rushing efficiency
# Key PFF columns (from RB/data_cleaning.py):
# ypa, yco_attempt, elusive_rating, mtf_per_attempt, breakaway_percent,
# explosive_rate, yprr, targets_per_route, fumble_rate

RB_FEATURES_WITH_COLLEGE = [
    # Basic athletic profile
    'Height', 'Weight', '40yd', 'Vertical', 'Bench', 'Broad Jump',
    # RAS + arm length if available
    'RAS', 'arm_length_inches',
    # Volume/efficiency rushing metrics
    'yards_after_contact', 'yco_attempt', 'ypa',
    # PFF-derived efficiency features requested for RBs
    'elusive_rating', 'mtf_per_attempt', 'breakaway_percent',
    'explosive_rate', 'yprr', 'targets_per_route', 'fumble_rate'
]

CONTAINS_WITH_COLLEGE_RB = [
    'contains_height', 'contains_weight', 'contains_40yd', 'contains_vertical',
    'contains_bench', 'contains_broad_jump',
    'contains_ras', 'contains_arm_length_inches',
    'contains_yards_after_contact',
    'contains_yco_attempt', 'contains_ypa',
    'contains_elusive_rating', 'contains_mtf_per_attempt', 'contains_breakaway_percent',
    'contains_explosive_rate', 'contains_yprr', 'contains_targets_per_route', 'contains_fumble_rate'
]

FEATURES_WITH_COLLEGE_ALL = RB_FEATURES_WITH_COLLEGE + CONTAINS_WITH_COLLEGE_RB

In [43]:
# Load RB training (2015–2023)
df = pd.read_csv('../data/processed/rb_training.csv')
df = df[df['Year'].between(2015, 2023)].copy()
print('Train (2015–2023 RBs):', len(df))

Train (2015–2023 RBs): 275


In [44]:
# Basic availability checks
total_count = len(df)
print(f"Players with RAS: {df['RAS'].notna().sum()} out of {total_count}")
print(f"Players with ypa: {df['ypa'].notna().sum()} out of {total_count}")
print(f"Players with yco_attempt: {df['yco_attempt'].notna().sum()} out of {total_count}")
print(f"Players with elusive_rating: {df['elusive_rating'].notna().sum()} out of {total_count}")
print(f"Players with mtf_per_attempt: {df['mtf_per_attempt'].notna().sum()} out of {total_count}")
print(f"Players with breakaway_percent: {df['breakaway_percent'].notna().sum()} out of {total_count}")
print(f"Players with explosive_rate: {df['explosive_rate'].notna().sum()} out of {total_count}")
print(f"Players with yprr: {df['yprr'].notna().sum()} out of {total_count}")
print(f"Players with targets_per_route: {df['targets_per_route'].notna().sum()} out of {total_count}")
print(f"Players with fumble_rate: {df['fumble_rate'].notna().sum()} out of {total_count}")

Players with RAS: 195 out of 275
Players with ypa: 222 out of 275
Players with yco_attempt: 222 out of 275
Players with elusive_rating: 222 out of 275
Players with mtf_per_attempt: 222 out of 275
Players with breakaway_percent: 222 out of 275
Players with explosive_rate: 222 out of 275
Players with yprr: 222 out of 275
Players with targets_per_route: 222 out of 275
Players with fumble_rate: 222 out of 275


In [45]:
# Helper: create "contains_" flags on the training dataframe
df['contains_height'] = df['Height'].notna().astype(int)
df['contains_weight'] = df['Weight'].notna().astype(int)
df['contains_40yd'] = df['40yd'].notna().astype(int)
df['contains_vertical'] = df['Vertical'].notna().astype(int)
df['contains_bench'] = df['Bench'].notna().astype(int)
df['contains_broad_jump'] = df['Broad Jump'].notna().astype(int)
df['contains_ras'] = df['RAS'].notna().astype(int)
df['contains_arm_length_inches'] = df['arm_length_inches'].notna().astype(int)

for col in ['attempts', 'yards', 'yards_after_contact', 'yco_attempt', 'ypa',
            'elusive_rating', 'mtf_per_attempt', 'breakaway_percent',
            'explosive_rate', 'yprr', 'targets_per_route', 'fumble_rate']:
    if col in df.columns:
        df[f'contains_{col}'] = df[col].notna().astype(int)
    else:
        df[f'contains_{col}'] = 0

In [46]:
# Target: round 1–8 (8 = undrafted)
y = np.where(df['Drafted'].astype(bool), np.clip(df['Round'].fillna(1).astype(int), 1, 7), 8)
X_raw = df[FEATURES_WITH_COLLEGE_ALL].copy()

imputer = KNNImputer(n_neighbors=10)
X = imputer.fit_transform(X_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_scaled, y)

y_pred_train = np.clip(ridge.predict(X_scaled), 1, 8)
print('Train MAE (round 1–8):', round(mean_absolute_error(y, y_pred_train), 4))
print('Train samples:', len(y))

Train MAE (round 1–8): 1.2516
Train samples: 275


In [47]:
# Prepare testing sets for 2024, 2025, 2026 from rb_testing.csv
rb_testing = pd.read_csv('../data/processed/rb_testing.csv') if os.path.exists('../data/processed/rb_testing.csv') else pd.DataFrame()
if rb_testing.empty:
    print('rb_testing.csv is empty; run RB/data_cleaning.py after creating rb_drafted_20xx.csv files.')
else:
    rb_2024 = rb_testing[rb_testing['Year'] == 2024].copy()
    rb_2025 = rb_testing[rb_testing['Year'] == 2025].copy()
    rb_2026 = rb_testing[rb_testing['Year'] == 2026].copy()
    print('RB 2024 rows:', len(rb_2024))
    print('RB 2025 rows:', len(rb_2025))
    print('RB 2026 rows:', len(rb_2026))

    def prepare_rb_df(ldf, year):
        """Mirror training transforms for RB test data (set Year, recompute contains_* flags)."""
        ldf = ldf.copy()
        ldf['Year'] = year
        # Ensure Height is numeric inches if any strings slipped in
        if ldf['Height'].dtype == object or (ldf['Height'].astype(str).str.contains('-', na=False).any()):
            def _height_inches(h):
                if pd.isna(h):
                    return np.nan
                s = str(h).strip()
                if '-' in s:
                    parts = s.split('-')
                    return int(parts[0]) * 12 + int(parts[1])
                try:
                    return float(s)
                except ValueError:
                    return np.nan
            ldf['Height'] = ldf['Height'].apply(_height_inches)
        else:
            ldf['Height'] = pd.to_numeric(ldf['Height'], errors='coerce')

        # contains_* flags – same pattern as training
        ldf['contains_height'] = ldf['Height'].notna().astype(int)
        ldf['contains_weight'] = ldf['Weight'].notna().astype(int)
        ldf['contains_40yd'] = ldf['40yd'].notna().astype(int)
        ldf['contains_vertical'] = ldf['Vertical'].notna().astype(int)
        ldf['contains_bench'] = ldf['Bench'].notna().astype(int)
        ldf['contains_broad_jump'] = ldf['Broad Jump'].notna().astype(int)
        ldf['contains_ras'] = ldf['RAS'].notna().astype(int) if 'RAS' in ldf.columns else 0
        ldf['contains_arm_length_inches'] = ldf['arm_length_inches'].notna().astype(int) if 'arm_length_inches' in ldf.columns else 0
        for col in ['yards_after_contact', 'yco_attempt', 'ypa',
                    'elusive_rating', 'mtf_per_attempt', 'breakaway_percent',
                    'explosive_rate', 'yprr', 'targets_per_route', 'fumble_rate']:
            if col in ldf.columns:
                ldf[f'contains_{col}'] = ldf[col].notna().astype(int)
            else:
                ldf[f'contains_{col}'] = 0
        return ldf

    # 2024 / 2025 evaluation
    rb_2024 = prepare_rb_df(rb_2024, 2024)
    rb_2025 = prepare_rb_df(rb_2025, 2025)

    X_24_raw = rb_2024[FEATURES_WITH_COLLEGE_ALL].copy()
    X_25_raw = rb_2025[FEATURES_WITH_COLLEGE_ALL].copy()
    X_24 = imputer.transform(X_24_raw)
    X_25 = imputer.transform(X_25_raw)
    X_24_scaled = scaler.transform(X_24)
    X_25_scaled = scaler.transform(X_25)

    pred_24 = np.clip(ridge.predict(X_24_scaled), 1, 8)
    pred_25 = np.clip(ridge.predict(X_25_scaled), 1, 8)

    actual_24 = rb_2024['Round'].astype(int).values
    actual_25 = rb_2025['Round'].astype(int).values

    def eval_metrics(actual, pred, label):
        mae = mean_absolute_error(actual, pred)
        rmse = np.sqrt(mean_squared_error(actual, pred))
        r2 = r2_score(actual, pred)
        exact = (np.round(pred) == actual).mean()
        within_1 = (np.abs(np.round(pred) - actual) <= 1).mean()
        print(f"{label} (n={len(actual)}): MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, Exact={exact:.2%}, Within-1={within_1:.2%}")

    print('2024 RBs:')
    eval_metrics(actual_24, pred_24, '2024')
    print('2025 RBs:')
    eval_metrics(actual_25, pred_25, '2025')

    # 2026 predictions table
    rb_2026 = prepare_rb_df(rb_2026, 2026)
    X_26_raw = rb_2026[FEATURES_WITH_COLLEGE_ALL].copy()
    X_26 = imputer.transform(X_26_raw)
    X_26_scaled = scaler.transform(X_26)
    pred_26 = np.clip(ridge.predict(X_26_scaled), 1, 8)

    def pred_round_to_tier(p):
        if p < 1.5: return ('1st', 'Elite')
        if p < 2.5: return ('2nd', 'Day 2')
        if p < 4.5: return ('3rd-4th', 'Day 2')
        if p < 6.5: return ('5th-6th', 'Day 3')
        if p < 7.5: return ('7th', 'Day 3')
        return ('UDFA', 'Undrafted')

    display_cols = [c for c in ['Round', 'Pick', 'Player', 'Pos', 'School', 'Year'] if c in rb_2026.columns]
    rb_2026_display = rb_2026[display_cols].copy()
    rb_2026_display['predicted_round'] = pred_26
    rb_2026_display['tier_label'] = [pred_round_to_tier(x)[0] for x in pred_26]

    rb_2026_display[['Player', 'School', 'predicted_round']].assign(Pos='RB').to_csv('../data/processed/rb_2026_predictions.csv', index=False)


RB 2024 rows: 18
RB 2025 rows: 23
RB 2026 rows: 12
2024 RBs:
2024 (n=18): MAE=1.4978, RMSE=1.8166, R²=-0.6006, Exact=16.67%, Within-1=61.11%
2025 RBs:
2025 (n=23): MAE=1.2773, RMSE=1.5413, R²=0.3627, Exact=30.43%, Within-1=52.17%


In [48]:
# 2024 & 2025 drafted RBs: table with prediction, tier, and interpretation

def rb_pred_round_to_tier(p):
    if p < 1.75: return ('Round 1 Tier', 'True 1st-round RB profile')
    if p < 2.75: return ('Round 2 Tier', 'Early Day 2 RB')
    if p < 3.75: return ('Round 3 Tier', 'Late Day 2 RB')
    if p < 4.75: return ('Round 4 Tier', 'Early Day 3 RB')
    if p < 5.75: return ('Round 5 Tier', 'Mid Day 3 RB')
    if p < 6.75: return ('Round 6 Tier', 'Late Day 3 RB')
    if p < 7.75: return ('Round 7 Tier', 'Fringe draftable RB')
    return ('UDFA Tier', 'UDFA / priority free agent')

rb_2024_display = rb_2024[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
rb_2024_display['predicted_round'] = pred_24
rb_2024_display['tier_label'] = [rb_pred_round_to_tier(x)[0] for x in pred_24]
rb_2024_display['interpretation'] = [rb_pred_round_to_tier(x)[1] for x in pred_24]
rb_2024_display['Round'] = rb_2024_display['Round'].astype(int)

rb_2025_display = rb_2025[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
rb_2025_display['predicted_round'] = pred_25
rb_2025_display['tier_label'] = [rb_pred_round_to_tier(x)[0] for x in pred_25]
rb_2025_display['interpretation'] = [rb_pred_round_to_tier(x)[1] for x in pred_25]
rb_2025_display['Round'] = rb_2025_display['Round'].astype(int)

print('2024 drafted RBs')
display(rb_2024_display)
print('2025 drafted RBs')
display(rb_2025_display)

2024 drafted RBs


,Round,Pick,Player,School,Year,predicted_round,tier_label,interpretation
0,4,120.0,Jaylen Wright,Tennessee,2024,2.049207,Round 2 Tier,Early Day 2 RB
1,2,46.0,Jonathon Brooks,Texas,2024,4.743210,Round 4 Tier,Early Day 3 RB
2,5,147.0,Audric Estime,Notre Dame,2024,3.674164,Round 3 Tier,Late Day 2 RB
3,6,173.0,Isaiah Davis,South Dakota State,2024,6.909275,Round 7 Tier,Fringe draftable RB
4,4,125.0,Bucky Irving,Oregon,2024,4.498556,Round 4 Tier,Early Day 3 RB
5,3,66.0,Trey Benson,Florida State,2024,3.757694,Round 4 Tier,Early Day 3 RB
6,6,181.0,Kimani Vidal,Troy,2024,2.775944,Round 3 Tier,Late Day 2 RB
7,3,88.0,MarShawn Lloyd,USC,2024,3.888108,Round 4 Tier,Early Day 3 RB
8,6,166.0,Tyrone Tracy Jr.,Purdue,2024,3.882750,Round 4 Tier,Early Day 3 RB
9,5,129.0,Isaac Guerendo,Louisville,2024,3.124945,Round 3 Tier,Late Day 2 RB


2025 drafted RBs


,Round,Pick,Player,School,Year,predicted_round,tier_label,interpretation
18,1,6.0,Ashton Jeanty,Boise State,2025,1.000000,Round 1 Tier,True 1st-round RB profile
19,1,22.0,Omarion Hampton,North Carolina,2025,1.344459,Round 1 Tier,True 1st-round RB profile
20,2,38.0,TreVeyon Henderson,Ohio State,2025,4.043147,Round 4 Tier,Early Day 3 RB
21,2,36.0,Quinshon Judkins,Ohio State,2025,3.847955,Round 4 Tier,Early Day 3 RB
22,3,83.0,Kaleb Johnson,Iowa,2025,3.267010,Round 3 Tier,Late Day 2 RB
23,4,126.0,Dylan Sampson,Tennessee,2025,4.120804,Round 4 Tier,Early Day 3 RB
24,4,105.0,Cam Skattebo,Arizona State,2025,1.832258,Round 2 Tier,Early Day 2 RB
25,6,184.0,Devin Neal,Kansas,2025,4.410478,Round 4 Tier,Early Day 3 RB
26,2,60.0,RJ Harvey,UCF,2025,4.658888,Round 4 Tier,Early Day 3 RB
27,7,223.0,Damien Martinez,Miami,2025,3.882177,Round 4 Tier,Early Day 3 RB


In [49]:
# 2026 RB predictions table with tier and interpretation (similar format to S notebook)

rb_2026_display_full = rb_2026[['Round', 'Pick', 'Player', 'School', 'Year']].copy()
rb_2026_display_full['predicted_round'] = pred_26
rb_2026_display_full['tier_label'] = [rb_pred_round_to_tier(x)[0] for x in pred_26]
rb_2026_display_full['interpretation'] = [rb_pred_round_to_tier(x)[1] for x in pred_26]

print(f'2026 RBs (n={len(pred_26)}): Predictions generated')
display(rb_2026_display_full.sort_values('predicted_round'))

# Save compact CSV for big_board_2026 (same schema as before)
rb_2026_display_full[['Player', 'School', 'predicted_round']].assign(Pos='RB').to_csv('../data/processed/rb_2026_predictions.csv', index=False)

2026 RBs (n=12): Predictions generated


,Round,Pick,Player,School,Year,predicted_round,tier_label,interpretation
41,NaN,NaN,Jeremiyah Love,Notre Dame,2026,2.369656,Round 2 Tier,Early Day 2 RB
46,NaN,NaN,Mike Washington Jr.,Arkansas,2026,2.964609,Round 3 Tier,Late Day 2 RB
47,NaN,NaN,Kaytron Allen,Penn State,2026,4.537702,Round 4 Tier,Early Day 3 RB
50,NaN,NaN,Seth McGowan,Kentucky,2026,4.729993,Round 4 Tier,Early Day 3 RB
49,NaN,NaN,J'Mari Taylor,Virginia,2026,4.773938,Round 5 Tier,Mid Day 3 RB
44,NaN,NaN,Emmett Johnson,Nebraska,2026,4.844352,Round 5 Tier,Mid Day 3 RB
45,NaN,NaN,Nicholas Singleton,Penn State,2026,5.193269,Round 5 Tier,Mid Day 3 RB
42,NaN,NaN,Jonah Coleman,Washington,2026,5.629957,Round 5 Tier,Mid Day 3 RB
43,NaN,NaN,Jadarian Price,Notre Dame,2026,5.874627,Round 6 Tier,Late Day 3 RB
48,NaN,NaN,Demond Claiborne,Wake,2026,7.270658,Round 7 Tier,Fringe draftable RB


In [50]:
# Model results on entire RB training set, ordered by predicted_round

if 'Pos' in df.columns:
    train_display = df[['Round', 'Pick', 'Player', 'Pos', 'School', 'Year']].copy()
else:
    train_display = df[['Round', 'Pick', 'Player', 'School', 'Year']].copy()

train_display['predicted_round'] = y_pred_train
train_display['tier_label'] = [rb_pred_round_to_tier(x)[0] for x in y_pred_train]
train_display['interpretation'] = [rb_pred_round_to_tier(x)[1] for x in y_pred_train]
train_display = train_display.sort_values('predicted_round').reset_index(drop=True)
train_display

,Round,Pick,Player,Pos,School,Year,predicted_round,tier_label,interpretation
0,2.0,45.0,Derrick Henry,RB,Alabama,2016,1.000000,Round 1 Tier,True 1st-round RB profile
1,2.0,41.0,Jonathan Taylor,RB,Wisconsin,2020,1.000000,Round 1 Tier,True 1st-round RB profile
2,1.0,27.0,Rashaad Penny,RB,San Diego State,2018,1.000000,Round 1 Tier,True 1st-round RB profile
3,2.0,62.0,AJ Dillon,RB,Boston College,2020,1.191375,Round 1 Tier,True 1st-round RB profile
4,1.0,8.0,Bijan Robinson,RB,Texas,2023,1.524816,Round 1 Tier,True 1st-round RB profile
...,...,...,...,...,...,...,...,...,...
270,NaN,NaN,Rakeem Boyd,RB,Arkansas,2021,8.000000,UDFA Tier,UDFA / priority free agent
271,NaN,NaN,Corey Dauphine,RB,Tulane,2021,8.000000,UDFA Tier,UDFA / priority free agent
272,NaN,NaN,Bryson Denley,RB,Bowling Green,2021,8.000000,UDFA Tier,UDFA / priority free agent
273,NaN,NaN,Mekhi Sargent,RB,Iowa,2021,8.000000,UDFA Tier,UDFA / priority free agent
